# Flood probabilities, frequencies and thresholds

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/03_probability_and_thresholds.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/03_probability_and_thresholds.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `cf47cba946da32dcbd99b8616d3db8b5b6fa4cb33ebd79655ad7692c92095974`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Flood probabilities, frequencies and thresholds

**Time:** 40 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Separate event rates from probabilities; calculate cumulative exposure; test how a threshold changes a headline count.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/03_probability_and_thresholds.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## Start with the unit

Annual exceedance probability $p$ is bounded by 0 and 1. An expected annual number of events $\lambda$ can exceed 1. The source workbook's EAE fields contain values above 1, so treating them directly as probability fractions is incorrect. Their full definitions, time windows and category rules need the original PAT metadata.

Under an explicitly assumed Poisson process, $P(N\geq1)=1-e^{-\lambda}$. That conversion is a model assumption, not a universal interpretation of an undocumented column.

For independent annual exceedance indicators with probabilities $p_y$, the probability of at least one event is $1-\prod_y(1-p_y)$. Independence may be unrealistic, and a changing climate makes constant annual probabilities a weak long-term assumption. A return period $T=1/p$ is an average recurrence concept under stationarity, not a timetable.

In [ ]:
p = 0.10
display(pd.DataFrame({'years':[1,10,30], 'at_least_one':[cumulative_probability([p]*n) for n in [1,10,30]]}))
rates=np.array([.01,.1,1,12])
display(pd.DataFrame({'assumed_Poisson_events_per_year':rates,'probability_at_least_one':1-np.exp(-rates)}))
assert np.isclose(cumulative_probability([.1]*10),1-.9**10)

## A classification is a decision rule

The fictional lab uses a logistic response: $p=1/(1+\exp((e-s-u)/a))$, with invented elevation $e$, sea-level increment $s$, surge offset $u=0.7$ m and scale $a=0.55$ m. This helps reveal how small height changes can affect a threshold. It has no calibration to the real portfolio.

The slider classifies a site as exposed when $p\geq\tau$. Changing $\tau$ changes the count even though the underlying probabilities have not changed. The denominator remains all facilities in the selected cohort.

In [ ]:
rows=select(facilities,exposure,scenario='RCP 8.5',year=2100)
thresholds=np.linspace(.01,.99,99)
fig=go.Figure(go.Scatter(x=thresholds,y=[cards(rows,t)['above_threshold'] for t in thresholds],mode='lines'))
fig.update_layout(title='Fictional count is sensitive to the decision threshold',xaxis_title='Annual probability threshold',yaxis_title='Facilities above threshold')
show(fig)
print('Sum of marginal probabilities (expected exposed facilities):',round(rows.annual_probability.sum(),2))
print('Threshold count:',cards(rows,.1)['above_threshold'])

In [ ]:
# Annual interpolation is explicit; nine decadal values are NOT nine decades of annual data.
example = exposure.query("facility_id == 'DEMO-01-001' and scenario == 'RCP 8.5'").sort_values('year')
annual_years=np.arange(2030,2061)
annual_p=np.interp(annual_years,example.year,example.annual_probability)
print('Illustrative 2030–2060 cumulative probability, independent years:',cumulative_probability(annual_p))

## Lab: interrogate uncertainty

Change the surge offset by ±0.25 m and compare counts at thresholds 1%, 10% and 50%. Then describe which is uncertainty about nature and which is a decision-maker's threshold.

**Worked check:** At fixed probabilities, increasing the threshold cannot increase the number classified as exposed. The sum of marginal probabilities is an expected number of exposed sites; it does not require independence between sites. Cumulative probability across years does require a dependence assumption for the product formula.

**Deliverable:** A sensitivity table with units and a paragraph stating the independence assumption. Do not add probabilities across years or interpret modeled exposed sites as affected patients.